In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

Matplotlib is building the font cache; this may take a moment.


In [2]:
# if I add or remove a ticker I need to change all 3 dictionaries
portfolio_weights = {
    "SPY": 0.30,  # S&P 500 ETF
    "AAPL": 0.10,  # Apple
    "MSFT": 0.10,  # Microsoft
    "JPM": 0.10,  # JPMorgan
    "XOM": 0.10,  # ExxonMobil
    "IEF": 0.20,  # 7-10 year US government bonds
    "GLD": 0.10  # gold
}

asset_names = {
    "SPY": "S&P 500 ETF",
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "JPM": "JPMorgan Chase",
    "XOM": "ExxonMobil",
    "IEF": "7-10 Year Treasury Bond ETF",
    "GLD": "Gold ETF"
}

asset_classes = {
    "SPY": "Equity",
    "AAPL": "Equity",
    "MSFT": "Equity",
    "JPM": "Equity",
    "XOM": "Equity",
    "IEF": "Bonds",
    "GLD": "Gold"
}

# yfinance doesn't include the end date so this gives data up to 31/12/2025
start_date = "2019-01-01"
end_date = "2026-01-01"

portfolio_value = 100000

In [3]:
# list of tickers in the same order as the dictionary
tickers = []
for ticker in portfolio_weights:
    tickers.append(ticker)

print("Number of assets:", len(tickers))
print(tickers)

total_weight = 0
for ticker in tickers:
    total_weight = total_weight + portfolio_weights[ticker]

# round because of tiny decimal errors (0.9999999999999999 instead of 1)
total_weight = round(total_weight, 4)
print("Total weight:", total_weight)

if total_weight != 1:
    print("WARNING - weights don't add up to 100%")

print()
print("Holdings:")
for ticker in tickers:
    weight = portfolio_weights[ticker]
    percent = round(weight * 100, 1)
    dollars = round(weight * portfolio_value)
    print(ticker, "-", asset_names[ticker], "-", percent, "% - $", dollars)

class_totals = {}
for ticker in tickers:
    asset_class = asset_classes[ticker]
    weight = portfolio_weights[ticker]
    if asset_class in class_totals:
        class_totals[asset_class] = class_totals[asset_class] + weight
    else:
        class_totals[asset_class] = weight

print()
print("By asset class:")
for asset_class in class_totals:
    percent = round(class_totals[asset_class] * 100, 1)
    print(asset_class, percent, "%")

Number of assets: 7
['SPY', 'AAPL', 'MSFT', 'JPM', 'XOM', 'IEF', 'GLD']
Total weight: 1.0

Holdings:
SPY - S&P 500 ETF - 30.0 % - $ 30000
AAPL - Apple - 10.0 % - $ 10000
MSFT - Microsoft - 10.0 % - $ 10000
JPM - JPMorgan Chase - 10.0 % - $ 10000
XOM - ExxonMobil - 10.0 % - $ 10000
IEF - 7-10 Year Treasury Bond ETF - 20.0 % - $ 20000
GLD - Gold ETF - 10.0 % - $ 10000

By asset class:
Equity 70.0 %
Bonds 20.0 %
Gold 10.0 %


In [ ]:
#downloading, cleaing and plotting data
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)

# auto_adjust=True means the Close prices are adjusted for dividends and splits
prices = data["Close"]

# yfinance puts the columns in alphabetical order so put them back in my order
prices = prices[tickers]

# saving a copy in case yfinance stops working
prices.to_csv("prices.csv")

[*********************100%***********************]  7 of 7 completed


In [7]:
print(prices.head())
print(prices.tail())

print("Rows and columns:", prices.shape)
print("First date:", prices.index[0])
print("Last date:", prices.index[-1])

print()
print("Missing values in each column:")
for ticker in tickers:
    missing = prices[ticker].isnull().sum()
    print(ticker, missing)

rows_before = len(prices)
prices = prices.dropna()
rows_after = len(prices)
print()
print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)

# a price of 0 or below would mean something is wrong with the data
print()
print("Lowest price of each asset:")
for ticker in tickers:
    lowest = prices[ticker].min()
    print(ticker, round(lowest, 2))

Ticker             SPY       AAPL       MSFT        JPM        XOM        IEF  \
Date                                                                            
2019-01-02  223.251572  37.436913  94.016151  80.074493  49.345222  86.240974   
2019-01-03  217.924194  33.707924  90.557472  78.936501  48.587605  86.926453   
2019-01-04  225.223679  35.146896  94.769264  81.846527  50.379002  86.224411   
2019-01-07  226.999527  35.068661  94.890106  81.903419  50.640991  85.984901   
2019-01-08  229.132263  35.737179  95.578125  81.748970  51.009178  85.770226   

Ticker             GLD  
Date                    
2019-01-02  121.330002  
2019-01-03  122.430000  
2019-01-04  121.440002  
2019-01-07  121.860001  
2019-01-08  121.529999  
Ticker             SPY        AAPL        MSFT         JPM         XOM  \
Date                                                                     
2025-12-24  685.029480  273.066711  484.943420  322.950073  116.875404   
2025-12-26  684.959961  272.657806 

In [ ]:

plt.figure(figsize=(12, 6))
for ticker in tickers:
    plt.plot(prices.index, prices[ticker], label=ticker)

plt.title("Adjusted closing prices")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()